# Data TU Mult 解析ノート

このノートブックは、PIMログから data_tu_mult ごとの衝突率と **1ノードあたりスループット** を解析し、PDFを出力します。

## 1. ライブラリの読み込み

解析と可視化に必要なライブラリを読み込みます。

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import japanize_matplotlib
import pandas as pd
import yaml

## 2. PDFの出力先設定

ログの入力先と、生成するPDFの保存先を設定します。必要に応じてこのセルだけ編集してください。

In [ ]:
# ログパス
PIM_LOG_DIR = Path("tsv/data_tu_mult_20260427")
TARGET_NODES = 10
TARGET_BACKOFF = 2048

# PDF出力先
OUTPUT_DIR = Path("img/analysis")
COLLISION_RATE_PDF = OUTPUT_DIR / "collision_rate_comparison_data_tu_mult_1_5.pdf"
THROUGHPUT_PER_NODE_BPS_PDF = OUTPUT_DIR / "throughput_per_node_bps_comparison_data_tu_mult_1_5.pdf"

## 3. 解析コード

data_tu_mult ごとの衝突率と **1ノードあたりスループット** を計算します。

In [ ]:
def load_config(config_path: Path) -> dict:
    """設定ファイルを読み込む。"""
    with config_path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def load_events(events_path: Path) -> pd.DataFrame:
    """events/evens TSVを読み込み、主要列を数値・文字列へ整形する。"""
    df = pd.read_csv(events_path, sep="\t")
    df["time_tu"] = pd.to_numeric(df["time_tu"], errors="coerce")
    df["round"] = pd.to_numeric(df["round"], errors="coerce").astype(int)
    df["event"] = df["event"].astype(str)
    df["result"] = df["result"].astype(str)
    return df


def parse_detail_fields(detail: str) -> dict:
    """detail文字列を key=value 形式の辞書へ変換する。"""
    out = {}
    for item in str(detail).split(","):
        part = item.strip()
        if "=" not in part:
            continue
        key, value = part.split("=", 1)
        out[key.strip()] = value.strip()
    return out


def iter_data_tu_mult_event_files(
    log_dir: Path,
    target_nodes: int = TARGET_NODES,
    target_backoff: int = TARGET_BACKOFF,
) -> list[tuple[int, Path]]:
    """events_10_2048_M.tsv / evens_10_2048_M.tsv を (data_tu_mult, path) で返す。"""
    files = []
    pattern = re.compile(r"eve(?:n)?ts_(\d+)_(\d+)_(\d+)\.tsv")
    for path in log_dir.glob("*.tsv"):
        m = pattern.fullmatch(path.name)
        if m is None:
            continue
        nodes = int(m.group(1))
        backoff = int(m.group(2))
        data_tu_mult = int(m.group(3))
        if nodes != target_nodes or backoff != target_backoff:
            continue
        files.append((data_tu_mult, path))
    return sorted(files, key=lambda x: x[0])


def summarize_data_tu_mult(
    log_dir: Path,
    target_nodes: int = TARGET_NODES,
    target_backoff: int = TARGET_BACKOFF,
) -> pd.DataFrame:
    """data_tu_mult ごとの衝突率と1ノードあたりスループット(bps)を集計して返す。"""
    config = load_config(log_dir / "config.yaml")
    data_bits = float(config["data_bits"])
    tu_sec = float(config["tu"])

    event_files = iter_data_tu_mult_event_files(log_dir, target_nodes, target_backoff)
    if not event_files:
        raise FileNotFoundError(
            f"events_{target_nodes}_{target_backoff}_M.tsv / evens_{target_nodes}_{target_backoff}_M.tsv が見つかりません: {log_dir}"
        )

    rows = []
    for data_tu_mult, events_path in event_files:
        events = load_events(events_path)
        if events.empty:
            continue

        successes = int((events["event"] == "rx_success").sum())
        collision_rows = events[events["event"] == "collision"]

        collision_packets = 0
        for detail in collision_rows["detail"]:
            fields = parse_detail_fields(detail)
            try:
                collision_packets += int(float(fields.get("count", "1")))
            except ValueError:
                collision_packets += 1

        attempts = successes + collision_packets
        collision_rate = (collision_packets / attempts) if attempts > 0 else 0.0

        round_last_tu = events.groupby("round")["time_tu"].max()
        num_rounds = int(round_last_tu.shape[0])
        avg_round_tu = float(round_last_tu.mean()) if num_rounds > 0 else 0.0
        avg_round_sec = avg_round_tu * tu_sec
        avg_successes_per_round = (successes / num_rounds) if num_rounds > 0 else 0.0

        total_throughput_bps = (
            (avg_successes_per_round * data_bits / avg_round_sec)
            if avg_round_sec > 0
            else 0.0
        )
        throughput_per_node_bps = (total_throughput_bps / target_nodes) if target_nodes > 0 else 0.0

        rows.append(
            {
                "nodes": target_nodes,
                "backoff": target_backoff,
                "data_tu_mult": data_tu_mult,
                "collisions": int(collision_rows.shape[0]),
                "successes": successes,
                "collision_rate": collision_rate,
                "throughput_per_node_bps": throughput_per_node_bps,
                "file": events_path.name,
            }
        )

    return pd.DataFrame(rows).sort_values("data_tu_mult").reset_index(drop=True)


def plot_collision_rate(df: pd.DataFrame, output_pdf: Path, nodes: int, backoff: int) -> None:
    """data_tu_multに対する衝突率をPDFに出力する。"""
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(df["data_tu_mult"], df["collision_rate"], marker="o")
    ax.set_title(f"Collision Rate vs Data TU Mult (nodes={nodes}, backoff={backoff})")
    ax.set_xlabel("data_tu_mult")
    ax.set_ylabel("Collision Rate")
    ax.set_xticks(df["data_tu_mult"])
    ax.set_ylim(0.0, 1.0)
    ax.grid(alpha=0.3)

    output_pdf.parent.mkdir(parents=True, exist_ok=True)
    with PdfPages(output_pdf) as pdf:
        pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


def plot_throughput_per_node_bps(df: pd.DataFrame, output_pdf: Path, nodes: int, backoff: int) -> None:
    """data_tu_multに対する1ノードあたりスループット(bps)をPDFに出力する。"""
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(df["data_tu_mult"], df["throughput_per_node_bps"], marker="o")
    ax.set_title(f"Throughput Per Node (bps) vs Data TU Mult (nodes={nodes}, backoff={backoff})")
    ax.set_xlabel("data_tu_mult")
    ax.set_ylabel("Throughput Per Node (bps)")
    ax.set_xticks(df["data_tu_mult"])
    ax.grid(alpha=0.3)

    output_pdf.parent.mkdir(parents=True, exist_ok=True)
    with PdfPages(output_pdf) as pdf:
        pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

## 4. 実行

このセルを実行すると、data_tu_mult解析を行ってPDFを出力し、結果テーブルを表示します。

In [ ]:
result = summarize_data_tu_mult(PIM_LOG_DIR, TARGET_NODES, TARGET_BACKOFF)

plot_collision_rate(result, COLLISION_RATE_PDF, nodes=TARGET_NODES, backoff=TARGET_BACKOFF)
plot_throughput_per_node_bps(
    result,
    THROUGHPUT_PER_NODE_BPS_PDF,
    nodes=TARGET_NODES,
    backoff=TARGET_BACKOFF,
)

print("data_tu_mult解析が完了しました。")
print(f"log dir: {PIM_LOG_DIR}")
print(f"collision PDF: {COLLISION_RATE_PDF}")
print(f"throughput per node PDF: {THROUGHPUT_PER_NODE_BPS_PDF}")
print()
print(f"=== 解析結果 (nodes={TARGET_NODES}, backoff={TARGET_BACKOFF}) ===")
display(result[["data_tu_mult", "collision_rate", "throughput_per_node_bps"]])